In [1]:
import torch
import pandas as pd
import torch
import cv2
import numpy as np

# set this to the directory where you download and unzip the full histopathologic-cancer-detection.zip file
data_dir = "/Users/ksankaran/Documents/data/"
df = pd.read_csv(f"{data_dir}/train_labels.csv")

In [2]:
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset

class HistopathologicDataset(Dataset):
    def __init__(self, df, datadir, transform=None):
        self.fnames = [f"{datadir}/{i}.tif" for i in df.id]
        self.labels = df.label.tolist()
        self.transform = transform
    
    def __len__(self):
        return len(self.fnames)
    
    def __getitem__(self, index):
        img = cv2.imread(self.fnames[index])
        if self.transform:
            img = self.transform(img)
        return img, self.labels[index], self.fnames[index]


normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.CenterCrop((49, 49)),
    transforms.ToTensor(),
    normalize,
])

valid_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.CenterCrop((49, 49)),
    transforms.ToTensor(),
    normalize,
])

In [3]:
split = int(0.8 * len(df))
batch_size = 64
train_dataset = HistopathologicDataset(df[:split], f"{data_dir}/train", train_transforms)
valid_dataset = HistopathologicDataset(df[split:], f"{data_dir}/train", valid_transforms)
train_loader = DataLoader(train_dataset, batch_size=batch_size)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

resnet = models.resnet50(pretrained=True)

from torch import nn
    
in_features = resnet.fc.in_features
num_hidden = 512

head = nn.Sequential(
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.BatchNorm1d(in_features),
    nn.Dropout(0.5),
    nn.Linear(in_features=in_features, out_features=num_hidden),
    nn.ReLU(),
    nn.BatchNorm1d(num_hidden),
    nn.Dropout(0.5),
    nn.Linear(in_features=num_hidden, out_features=2),
)

model = nn.Sequential(
    nn.Sequential(*list(resnet.children())[:-2]),
    head
)


In [4]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

def train(train_loader, model, criterion, optimizer):
    total_loss = 0.0
    size = len(train_loader.dataset)
    num_batches = size // train_loader.batch_size
    model.train()
    for i, (images, labels, file) in enumerate(train_loader):
        print(f"Training: {i}/{num_batches}", end="\r")
        
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images) # forward pass
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        loss.backward()  # backprogagation
        optimizer.step()
        
    return total_loss / size

def validate(valid_loader, model, criterion):
    model.eval()
    with torch.no_grad():
        total_correct = 0
        total_loss = 0.0
        size = len(valid_loader.dataset)
        num_batches = size // valid_loader.batch_size
        for i, (images, labels, file) in enumerate(valid_loader):
            print(f"Validation: {i}/{num_batches}", end="\r")
            
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            total_correct += torch.sum(preds == labels.data)
            total_loss += loss.item() * images.size(0)
            
        return total_loss / size, total_correct.double() / size


def fit(model, num_epochs, train_loader, valid_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
    print("epoch\ttrain loss\tvalid loss\taccuracy")
    for epoch in range(num_epochs):
        train_loss = train(train_loader, model, criterion, optimizer)
        valid_loss, valid_acc = validate(valid_loader, model, criterion)
        print(f"{epoch}\t{train_loss:.5f}\t\t{valid_loss:.5f}\t\t{valid_acc:.3f}")


In [5]:
model = model.to(device)
#fit(model, 1, train_loader, valid_loader)
#torch.save(model.state_dict(), "/Users/ksankaran/Documents/state.pt")

In [6]:
state = torch.load("/Users/ksankaran/Documents/state.pt")
model.load_state_dict(state)


# to get activations after layer 7 (just before the linear classification), we
# create a submodel up to that layer and pass in a subsample. We save the activations
# to a large numpy array which will then be converted to a pandas data.frame and then
# written to file.

<All keys matched successfully>

In [10]:
from pathlib import Path

activation_model = nn.Sequential(model[0], model[1][:-1])

h, fnames, ys = [], [], []
for i in range(2):
    x, y, f = next(iter(valid_loader))
    h.append(activation_model(x))
    fnames.append([Path(s).name for s in f])
    ys.append(y)

h = torch.cat(h)
fnames = sum(fnames, [])
y = torch.cat(ys)

In [11]:
import numpy as np

np.save("/Users/ksankaran/Downloads/h.npy", h.detach().numpy())
np.save("/Users/ksankaran/Downloads/y.npy", y)
np.save("/Users/ksankaran/Downloads/fnames.npy", np.array(fnames))